# Notebook 01b — Lexicon-Based Baselines

**Project:** Benchmarking Modern Multilingual Transformer Models for Malay-English Code-Switched Sentiment Analysis  
**Authors:** Joshua Joenathan Thomas (25141571) | Amit Kumar Gupta (25109952)  

## What this notebook does
Evaluates three off-the-shelf lexicon-based sentiment tools on the gold test set (n=2,000, manual labels).  
These are **zero-shot / unsupervised** — no fine-tuning, no training data used.

| Tool | Type | Language coverage |
|------|------|------------------|
| VADER | Rule-based lexicon (valence-aware) | English |
| TextBlob | Pattern-based lexicon | English |
| pysentimiento | Pretrained RoBERTa-based multilingual | 10+ languages |

**Expected finding:** All three will perform poorly on code-switched Malay-English text because:  
1. VADER and TextBlob are English-only — Malay tokens are invisible to them  
2. pysentimiento does not include Malay in its training languages  

This poor performance is itself a meaningful result — it motivates fine-tuned multilingual transformers.

In [ ]:
import sys
import sys
from pathlib import Path
_cwd = Path.cwd()
for _p in [_cwd / '../src', _cwd / '../3_Source', _cwd / 'src', _cwd / '3_Source']:
    if (_p / 'config.py').exists():
        sys.path.insert(0, str(_p.resolve()))
        break
else:
    raise ImportError('config.py not found -- run Jupyter from the project root or notebooks/ directory')

import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from config import TEST_CSV, RESULTS_DIR, SEED

(RESULTS_DIR / 'lexicon').mkdir(parents=True, exist_ok=True)

print('Config loaded.')

In [ ]:
# Load gold test set (manual labels only)
df_test_raw = pd.read_csv(TEST_CSV)
df_test_raw.columns = ['text', 'label_auto', 'label_manual']
df_test = df_test_raw[['text', 'label_manual']].copy()
df_test['label'] = df_test['label_manual'].str.upper().str.strip()
df_test['text_clean'] = df_test['text'].apply(
    lambda t: re.sub(r'http\S+|www\.\S+', '', str(t).lower()).strip()
)

print(f'Test set: {len(df_test):,} samples')
print(f'Class distribution: {dict(df_test["label"].value_counts())}')

## 1. VADER

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

sia = SentimentIntensityAnalyzer()

def vader_predict(text):
    scores = sia.polarity_scores(str(text))
    compound = scores['compound']
    # Standard VADER thresholds
    if compound >= 0.05:
        return 'POSITIVE'
    elif compound <= -0.05:
        return 'NEGATIVE'
    else:
        return 'NEUTRAL'

df_test['vader_pred'] = df_test['text_clean'].apply(vader_predict)

label_order = ['NEGATIVE', 'NEUTRAL', 'POSITIVE']
vader_macro_f1 = f1_score(df_test['label'], df_test['vader_pred'], average='macro', labels=label_order)
vader_report = classification_report(
    df_test['label'], df_test['vader_pred'],
    labels=label_order, target_names=label_order, output_dict=True
)

print(f'=== VADER — Gold Test Set ===')
print(f'Macro-F1: {vader_macro_f1:.4f}')
print(classification_report(df_test['label'], df_test['vader_pred'],
                             labels=label_order, target_names=label_order))
print(f'Prediction distribution: {dict(df_test["vader_pred"].value_counts())}')

In [ ]:
# VADER confusion matrix
cm = confusion_matrix(df_test['label'], df_test['vader_pred'], labels=label_order)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=label_order).plot(ax=ax, cmap='Blues')
ax.set_title(f'VADER — Gold Test Set (Macro-F1 = {vader_macro_f1:.4f})', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lexicon' / 'vader_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

vader_results = {
    'model': 'VADER',
    'type': 'lexicon',
    'macro_f1': round(vader_macro_f1, 4),
    'per_class': {cls: {
        'precision': round(vader_report[cls]['precision'], 4),
        'recall':    round(vader_report[cls]['recall'], 4),
        'f1':        round(vader_report[cls]['f1-score'], 4)
    } for cls in label_order}
}
with open(RESULTS_DIR / 'lexicon' / 'vader_results.json', 'w') as f:
    json.dump(vader_results, f, indent=2)
print('Saved: results/lexicon/vader_results.json')

## 2. TextBlob

In [ ]:
from textblob import TextBlob

def textblob_predict(text):
    polarity = TextBlob(str(text)).sentiment.polarity
    if polarity > 0.05:
        return 'POSITIVE'
    elif polarity < -0.05:
        return 'NEGATIVE'
    else:
        return 'NEUTRAL'

df_test['textblob_pred'] = df_test['text_clean'].apply(textblob_predict)

tb_macro_f1 = f1_score(df_test['label'], df_test['textblob_pred'], average='macro', labels=label_order)
tb_report = classification_report(
    df_test['label'], df_test['textblob_pred'],
    labels=label_order, target_names=label_order, output_dict=True
)

print(f'=== TextBlob — Gold Test Set ===')
print(f'Macro-F1: {tb_macro_f1:.4f}')
print(classification_report(df_test['label'], df_test['textblob_pred'],
                             labels=label_order, target_names=label_order))
print(f'Prediction distribution: {dict(df_test["textblob_pred"].value_counts())}')

In [ ]:
cm = confusion_matrix(df_test['label'], df_test['textblob_pred'], labels=label_order)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=label_order).plot(ax=ax, cmap='Blues')
ax.set_title(f'TextBlob — Gold Test Set (Macro-F1 = {tb_macro_f1:.4f})', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lexicon' / 'textblob_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

tb_results = {
    'model': 'TextBlob',
    'type': 'lexicon',
    'macro_f1': round(tb_macro_f1, 4),
    'per_class': {cls: {
        'precision': round(tb_report[cls]['precision'], 4),
        'recall':    round(tb_report[cls]['recall'], 4),
        'f1':        round(tb_report[cls]['f1-score'], 4)
    } for cls in label_order}
}
with open(RESULTS_DIR / 'lexicon' / 'textblob_results.json', 'w') as f:
    json.dump(tb_results, f, indent=2)
print('Saved: results/lexicon/textblob_results.json')

## 3. pysentimiento

pysentimiento uses a fine-tuned RoBERTa-based model for sentiment analysis. It supports Spanish, English, Italian, Portuguese, and Arabic — **not Malay**. Applied here to assess whether a multilingual pretrained model without Malay generalises to code-switched text.

In [ ]:
from pysentimiento import create_analyzer

# English analyzer — closest available to code-switched Malay-English
print('Loading pysentimiento English analyzer...')
analyzer = create_analyzer(task='sentiment', lang='en')
print('Loaded.')

# Map pysentimiento labels to our schema
PYSEN_MAP = {'POS': 'POSITIVE', 'NEG': 'NEGATIVE', 'NEU': 'NEUTRAL'}

def pysentimiento_predict(text):
    result = analyzer.predict(str(text)[:512])   # truncate for safety
    return PYSEN_MAP.get(result.output, 'NEUTRAL')

print('Running inference on 2,000 test samples...')
df_test['pysen_pred'] = df_test['text_clean'].apply(pysentimiento_predict)
print('Done.')

In [ ]:
pysen_macro_f1 = f1_score(df_test['label'], df_test['pysen_pred'], average='macro', labels=label_order)
pysen_report = classification_report(
    df_test['label'], df_test['pysen_pred'],
    labels=label_order, target_names=label_order, output_dict=True
)

print(f'=== pysentimiento (en) — Gold Test Set ===')
print(f'Macro-F1: {pysen_macro_f1:.4f}')
print(classification_report(df_test['label'], df_test['pysen_pred'],
                             labels=label_order, target_names=label_order))
print(f'Prediction distribution: {dict(df_test["pysen_pred"].value_counts())}')

cm = confusion_matrix(df_test['label'], df_test['pysen_pred'], labels=label_order)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=label_order).plot(ax=ax, cmap='Blues')
ax.set_title(f'pysentimiento (en) — Gold Test Set (Macro-F1 = {pysen_macro_f1:.4f})', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lexicon' / 'pysentimiento_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

pysen_results = {
    'model': 'pysentimiento (en)',
    'type': 'lexicon',
    'macro_f1': round(pysen_macro_f1, 4),
    'per_class': {cls: {
        'precision': round(pysen_report[cls]['precision'], 4),
        'recall':    round(pysen_report[cls]['recall'], 4),
        'f1':        round(pysen_report[cls]['f1-score'], 4)
    } for cls in label_order}
}
with open(RESULTS_DIR / 'lexicon' / 'pysentimiento_results.json', 'w') as f:
    json.dump(pysen_results, f, indent=2)
print('Saved: results/lexicon/pysentimiento_results.json')

## 4. Summary — All Lexicon Models

In [ ]:
print('=== LEXICON BASELINES SUMMARY ===')
print(f'{"Model":<25} {"Macro-F1":>9} {"NEG-F1":>8} {"NEU-F1":>8} {"POS-F1":>8}')
print('-' * 62)
for res in [vader_results, tb_results, pysen_results]:
    print(f'{res["model"]:<25} {res["macro_f1"]:>9.4f}',
          f'{res["per_class"]["NEGATIVE"]["f1"]:>8.4f}',
          f'{res["per_class"]["NEUTRAL"]["f1"]:>8.4f}',
          f'{res["per_class"]["POSITIVE"]["f1"]:>8.4f}')

# Save combined lexicon results
all_lexicon = [vader_results, tb_results, pysen_results]
with open(RESULTS_DIR / 'lexicon' / 'all_lexicon_results.json', 'w') as f:
    json.dump(all_lexicon, f, indent=2)
print('\nSaved: results/lexicon/all_lexicon_results.json')
print()
print('Key finding: All lexicon models perform near or below random chance on code-switched')
print('Malay-English text. VADER and TextBlob are English-only; they cannot interpret Malay')
print('tokens. pysentimiento, despite being RoBERTa-based, was not trained on Malay.')
print('This provides strong motivation for fine-tuned multilingual transformers.')

In [ ]:
# V2 Fix: Re-evaluate all lexicons on 1,700-sample clean test set
# All v2 transformer experiments use CLEAN_TEST_CSV. Lexicons must use the same set.
from config import CLEAN_TEST_CSV, RESULTS_DIR as RD
import json as _json

df_ct = pd.read_csv(CLEAN_TEST_CSV)
df_ct["text"] = df_ct["text"].str.strip()
# Apply same preprocessing as v1 (lowercase + strip URLs)
df_ct["text_clean"] = df_ct["text"].apply(
    lambda t: re.sub(r"http\S+|www\.\S+", "", str(t).lower()).strip()
)

# Re-apply all three classifiers (functions defined in earlier cells)
df_ct["vader_pred"]    = df_ct["text_clean"].apply(vader_predict)
df_ct["textblob_pred"] = df_ct["text_clean"].apply(textblob_predict)
df_ct["pysen_pred"]    = df_ct["text_clean"].apply(pysentimiento_predict)

label_order = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
lexicon_v2_results = {}

for key, col, name in [
    ("vader_v2",         "vader_pred",   "VADER"),
    ("textblob_v2",      "textblob_pred", "TextBlob"),
    ("pysentimiento_v2", "pysen_pred",    "pysentimiento (en)"),
]:
    mf1 = f1_score(df_ct["label"], df_ct[col], average="macro")
    pc  = {lbl: round(f1_score(df_ct["label"], df_ct[col], labels=[lbl], average="macro"), 4)
           for lbl in label_order}
    lexicon_v2_results[key] = {
        "model": name, "macro_f1": round(mf1, 4),
        "per_class_f1": pc, "test_set": "clean_test_1700"
    }
    print(f"{name:<25} Macro-F1 (1700-sample clean test): {mf1:.4f}  | "
          f"NEG={pc['NEGATIVE']:.4f}  NEU={pc['NEUTRAL']:.4f}  POS={pc['POSITIVE']:.4f}")

out_path = RD / "lexicons" / "lexicon_v2_clean_test_results.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w") as f:
    _json.dump(lexicon_v2_results, f, indent=2)
print(f"Saved: {out_path}")
